In [1]:

import sqlite3
import json
from pathlib import Path

# Database location
DB = Path("work/personal/catalogue.sqlite")

BATCH_IDS = [
    "batch-3af58a28379ac9617ce37aabfaef8cecc098fb8a8fff7189e4203348c862e2c8",
    "batch-b8bd7bbd6af9f896c778f6eed3d7843432e3ce007babdec1ea54e884e7f20baa"
]

# Open database in read-only mode
connection = sqlite3.connect(
    DB.resolve().as_uri() + "?mode=ro",
    uri=True
)
connection.row_factory = sqlite3.Row

results = []

try:
    for batch_id in BATCH_IDS:
        members = connection.execute("""
            SELECT
                batch_id,
                member_id,
                planned_request_id,
                spec_json
            FROM batch_members
            WHERE batch_id = ?
        """, (batch_id,)).fetchall()

        for member in members:
            request_id = member["planned_request_id"]
            spec = json.loads(member["spec_json"])

            attempts = connection.execute("""
                SELECT attempt_id, status, stage
                FROM processing_attempts
                WHERE request_id = ?
                ORDER BY started_at
            """, (request_id,)).fetchall()

            result = {
                "batch_id": batch_id,
                "member_id": member["member_id"],
                "request_id": request_id,
                "epsilon": spec["parameters"]["epsilon"],
                "attempts": [dict(a) for a in attempts]
            }

            results.append(result)

finally:
    connection.close()

print("\n========== REQUEST COMPARISON ==========\n")

for result in results:
    print("Batch ID:", result["batch_id"])
    print("Member:", result["member_id"])
    print("Request ID:", result["request_id"])
    print("Epsilon:", result["epsilon"])
    print("Attempts:")

    for attempt in result["attempts"]:
        print("  Attempt ID:", attempt["attempt_id"])
        print("  Status:", attempt["status"])

    print("-" * 60)

# Automatic verification
if len(results) >= 2:
    first, second = results[0], results[1]

    print("\n========== VERIFICATION ==========\n")

    print("Different request IDs:",
          first["request_id"] != second["request_id"])

    print("Different epsilon values:",
          first["epsilon"] != second["epsilon"])

    print("Different attempt IDs:",
          first["attempts"][0]["attempt_id"]
          != second["attempts"][0]["attempt_id"])

    print("Both succeeded:",
          all(
              a["status"] == "succeeded"
              for r in results[:2]
              for a in r["attempts"]
          ))

    if (
        first["request_id"] != second["request_id"]
        and first["epsilon"] != second["epsilon"]
        and first["attempts"][0]["attempt_id"]
        != second["attempts"][0]["attempt_id"]
    ):
        print("\nSUCCESS: New scientific request verified!")
    else:
        print("\nWARNING: Some conditions were not satisfied.")

else:
    print("\nERROR: Expected at least two batch members.")



========== REQUEST COMPARISON ==========

Batch ID: batch-3af58a28379ac9617ce37aabfaef8cecc098fb8a8fff7189e4203348c862e2c8
Member: las_wolski_ndmi_20250605
Request ID: request-e421453eb810d87ba198a3db277c6441a2f5f256db34fdd05c263cbdb9e99fb3
Epsilon: 1e-06
Attempts:
  Attempt ID: attempt-1c6cd889eb154dc3bd837b24a51617de
  Status: succeeded
------------------------------------------------------------
Batch ID: batch-b8bd7bbd6af9f896c778f6eed3d7843432e3ce007babdec1ea54e884e7f20baa
Member: las_wolski_ndmi_20250605
Request ID: request-4d6e40d5ded7449aab3925de5ea6697bff226a90ac34a49842b9b22290fe3f9d
Epsilon: 1e-05
Attempts:
  Attempt ID: attempt-d69c13390bc44f09af4045cacbc35a9d
  Status: succeeded
------------------------------------------------------------

========== VERIFICATION ==========

Different request IDs: True
Different epsilon values: True
Different attempt IDs: True
Both succeeded: True

SUCCESS: New scientific request verified!


In [11]:

import json
from pathlib import Path

import numpy as np
import rasterio

from eo_processing import database as db
from eo_processing.planning import load_manifest
from eo_processing.preparation import prepare, inspect, warp


MANIFEST = Path("work/personal/batch_v3.json")
REPORT = Path("work/personal/report/report_v3_check.json/report.json")

manifest = load_manifest(MANIFEST)
report = json.loads(REPORT.read_text(encoding="utf-8"))

member = next(
    m for m in manifest["frozen"]["members"]
    if m["member_id"] == "las_wolski_ndmi_20250605"
)

spec = member["spec"]

# Retrieve the original registered input assets.
with db.connect(manifest["runtime"]["database"], readonly=True) as conn:
    paths = {
        role: db.location(conn, record["artifact_id"])
        for role, record in spec["inputs"].items()
    }

# Reconstruct the prepared reflectance arrays.
bands, valid, inside, grid, metrics = prepare(
    spec["inputs"],
    paths,
    spec["preparation"],
    spec["product"]["required_bands"]
)

# Locate the actual NDMI output.
report_member = next(
    m for m in report["members"]
    if m["member_id"] == member["member_id"]
)

output_path = report_member["result"]["paths"]["index"]

with rasterio.open(output_path) as src:
    stored = src.read(1)

nir = bands["nir08"]
swir = bands["swir16"]
epsilon = spec["parameters"]["epsilon"]

# Independent calculation of the NDMI formula.
denominator = nir + swir

usable = (
    valid
    & np.isfinite(nir)
    & np.isfinite(swir)
    & (np.abs(denominator) > epsilon)
)

manual = np.full(nir.shape, np.nan, dtype="float32")

manual[usable] = (
    (nir[usable] - swir[usable])
    / denominator[usable]
)

# Compare the independently calculated result with the GeoTIFF.
difference = np.abs(manual[usable] - stored[usable])

print("\n========== NDMI PIXEL CHECK ==========")

print("Usable pixels:", int(usable.sum()))
print("Maximum absolute difference:", float(difference.max()))
print("Mean absolute difference:", float(difference.mean()))

print("\nSample pixels:")

coordinates = np.argwhere(usable)

for position in np.linspace(
    0, len(coordinates) - 1,
    min(5, len(coordinates)),
    dtype=int
):
    row, col = coordinates[position]

    print(
        f"Row={row}, Col={col}, "
        f"NIR={nir[row,col]:.6f}, "
        f"SWIR={swir[row,col]:.6f}, "
        f"Manual={manual[row,col]:.6f}, "
        f"Raster={stored[row,col]:.6f}"
    )

# Check that cloud pixels remain NaN.
scl, scl_mask = inspect(paths["scl"], spec["inputs"]["scl"])
scl = np.where(scl_mask, scl, 0).astype("uint8")
target_scl = warp(
    scl,
    spec["inputs"]["scl"]["raster"],
    grid,
    categorical=True
)

cloud_pixels = inside & np.isin(target_scl, [3, 8, 9, 10])
cloud_count = int(cloud_pixels.sum())
cloud_finite = int(np.isfinite(stored[cloud_pixels]).sum())

print("\n========== CLOUD MASK CHECK ==========")
print("Cloud pixels inside AOI:", cloud_count)
print("Cloud pixels with finite NDMI:", cloud_finite)

print("\nIndependent check:",
      "PASSED" if difference.max() < 1e-7 else "REQUIRES INVESTIGATION")

print("Cloud mask:",
      "PASSED" if cloud_finite == 0 else "REQUIRES INVESTIGATION")



========== NDMI PIXEL CHECK ==========
Usable pixels: 11751
Maximum absolute difference: 5.960464477539063e-08
Mean absolute difference: 1.1052243564790842e-08

Sample pixels:
Row=1, Col=3, NIR=0.449500, SWIR=0.179300, Manual=0.429707, Raster=0.429707
Row=28, Col=80, NIR=0.386000, SWIR=0.167800, Manual=0.394005, Raster=0.394005
Row=56, Col=21, NIR=0.419400, SWIR=0.179000, Manual=0.401738, Raster=0.401738
Row=83, Col=52, NIR=0.352600, SWIR=0.164200, Manual=0.364551, Raster=0.364551
Row=113, Col=107, NIR=0.244500, SWIR=0.227100, Manual=0.036896, Raster=0.036896

========== CLOUD MASK CHECK ==========
Cloud pixels inside AOI: 0
Cloud pixels with finite NDMI: 0

Independent check: PASSED
Cloud mask: PASSED
